In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class ResNetBackbone(nn.Module):
    def __init__(self):
        super(ResNetBackbone, self).__init__()
        resnet = models.resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # Keep layers except for the last two (FC and pool)
    
    def forward(self, x):
        return self.backbone(x)


# batch normalisation is not done
class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ASPP, self).__init__()
        self.aspp1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, dilation=1, bias=False)
        self.aspp2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=6, dilation=6, bias=False)
        self.aspp3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=12, dilation=12, bias=False)
        self.aspp4 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=18, dilation=18, bias=False)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.final_conv = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.relu = nn.ReLU()

    def forward(self, x):
        aspp1 = self.relu(self.aspp1(x))
        aspp2 = self.relu(self.aspp2(x))
        aspp3 = self.relu(self.aspp3(x))
        aspp4 = self.relu(self.aspp4(x))
        global_avg_pool = self.global_avg_pool(x)
        global_avg_pool = self.conv1x1(global_avg_pool)
        global_avg_pool = nn.Upsample(size=aspp1.shape[2:], mode='bilinear', align_corners=True)(global_avg_pool)
        aspp_output = torch.cat([aspp1, aspp2, aspp3, aspp4, global_avg_pool], dim=1)
        return self.final_conv(aspp_output)


class DeepLabDecoder(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(DeepLabDecoder, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 256, kernel_size=3, stride=1, padding=1, bias=False)
        self.conv2 = nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1, bias=False)
        self.final_conv = nn.Conv2d(256, num_classes, kernel_size=1, stride=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True)(self.final_conv(x))


class DeepLabV3(nn.Module):
    def __init__(self, num_classes):
        super(DeepLabV3, self).__init__()
        self.backbone = ResNetBackbone()
        self.aspp = ASPP(2048, 256)
        self.decoder = DeepLabDecoder(256, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        aspp_output = self.aspp(features)
        return self.decoder(aspp_output)


In [ ]:
model = DeepLabV3(num_classes=24)  # Example for 24 classes
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
input_image = torch.randn(1, 3, 512, 512)  # Example input tensor
output = model(input_image)
print(output.shape)  # Expected output shape: (1, 24, 512, 512)